In [ ]:
"""
Numerical example: relaxing a categorical distribution into the simplex.

Sequence: "cat" over vocabulary {c, a, t, o}  (K = 4)
"""

import numpy as np

np.set_printoptions(precision=3, suppress=True)

# ---------------------------------------------------------------
# Step 0: define vocabulary
# ---------------------------------------------------------------
vocab = ['c', 'a', 't', 'o']
K = len(vocab)
char_to_idx = {ch: i for i, ch in enumerate(vocab)}

print("=" * 60)
print("STEP 0 — Vocabulary")
print("=" * 60)
print(f"vocab     = {vocab}")
print(f"K         = {K}")
print(f"char->idx = {char_to_idx}")

# ---------------------------------------------------------------
# Step 1: tokenize the sequence "cat"
# ---------------------------------------------------------------
sequence = "cat"
indices = [char_to_idx[ch] for ch in sequence]

print()
print("=" * 60)
print("STEP 1 — Tokenize")
print("=" * 60)
print(f"sequence = {sequence!r}")
print(f"indices  = {indices}    (these are integers in {{0,...,{K-1}}})")

# ---------------------------------------------------------------
# Step 2: one-hot encode -> this IS x
# ---------------------------------------------------------------
# Standard basis vectors e_0, e_1, e_2, e_3 of R^4
I = np.eye(K)
print()
print("=" * 60)
print("STEP 2 — One-hot encode (this gives us x)")
print("=" * 60)
print("Standard basis vectors of R^K (the simplex vertices e_i):")
for i in range(K):
    print(f"  e_{i} ('{vocab[i]}') = {I[i]}")

x = np.stack([I[i] for i in indices])  # shape (3, K)
print()
print("x = sequence of one-hot vectors, one per token:")
for t, (ch, vec) in enumerate(zip(sequence, x)):
    print(f"  position {t} ('{ch}'): {vec}")
print(f"x.shape = {x.shape}    (3 tokens, each a vector in R^{K})")

# ---------------------------------------------------------------
# Step 3: describe the per-position distribution
# ---------------------------------------------------------------
# For a SINGLE deterministic token "a" (position 1), the distribution is
# just a delta at e_1. To make the mixture-of-deltas form non-trivial,
# pretend position 1 has empirical probabilities over the vocab.
print()
print("=" * 60)
print("STEP 3 — Describe distribution at one position as delta mixture")
print("=" * 60)
p = np.array([0.1, 0.7, 0.0, 0.2])
print(f"Suppose position 1 has categorical probs p = {p}")
print(f"  (sum = {p.sum()})")
print()
print("p_data(x) = sum_i p_i * delta(x - e_i)")
print("         =", " + ".join(
    f"{p[i]}*delta(x - e_{i})" for i in range(K) if p[i] > 0
))
print()
print("Sampling from this distribution returns one of these vectors:")
for i in range(K):
    if p[i] > 0:
        print(f"  e_{i} = {I[i]}   with probability {p[i]}")

# ---------------------------------------------------------------
# Step 4: continuous-space operation — add Gaussian noise
# ---------------------------------------------------------------
print()
print("=" * 60)
print("STEP 4 — Why the continuous view matters: add Gaussian noise")
print("=" * 60)

rng = np.random.default_rng(seed=0)
sigma = 0.1

# Take the clean "a" token = e_1
x_clean = I[1].copy()
print(f"Clean one-hot for 'a': x = {x_clean}")
print(f"Noise scale sigma = {sigma}")

eps = rng.standard_normal(K)
x_noisy = x_clean + sigma * eps
print(f"epsilon ~ N(0, I) sample: {eps}")
print(f"x_noisy = e_1 + sigma * epsilon =")
print(f"          {x_noisy}")

print()
print("Sanity checks on x_noisy:")
print(f"  sums to 1?       {np.isclose(x_noisy.sum(), 1.0)}  (sum = {x_noisy.sum():.4f})")
print(f"  all nonneg?      {np.all(x_noisy >= 0)}  (min = {x_noisy.min():.4f})")
print(f"  is a one-hot?    {np.any(np.allclose(x_noisy, I, atol=1e-6))}")
print("  => x_noisy left the simplex; it's a generic point in R^K.")
print("     But its 2nd entry is still ~1, so it 'points toward' e_1.")

# ---------------------------------------------------------------
# Summary table
# ---------------------------------------------------------------
print()
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"{'stage':<22}{'object':<32}{'lives in'}")
print("-" * 70)
print(f"{'raw text':<22}{repr(sequence):<32}strings")
print(f"{'token indices':<22}{str(indices):<32}{{0,...,{K-1}}}^3")
print(f"{'one-hot (= x)':<22}{'(e_0, e_1, e_2)':<32}vertices of S_{K} in R^{K}")
print(f"{'delta-mixture form':<22}{'sum_i p_i delta(x - e_i)':<32}measure on R^{K}")
print(f"{'noised':<22}{'e_i + sigma*epsilon':<32}generic point in R^{K}")

STEP 0 — Vocabulary
vocab     = ['c', 'a', 't', 'o']
K         = 4
char->idx = {'c': 0, 'a': 1, 't': 2, 'o': 3}

STEP 1 — Tokenize
sequence = 'cat'
indices  = [0, 1, 2]    (these are integers in {0,...,3})

STEP 2 — One-hot encode (this gives us x)
Standard basis vectors of R^K (the simplex vertices e_i):
  e_0 ('c') = [1. 0. 0. 0.]
  e_1 ('a') = [0. 1. 0. 0.]
  e_2 ('t') = [0. 0. 1. 0.]
  e_3 ('o') = [0. 0. 0. 1.]

x = sequence of one-hot vectors, one per token:
  position 0 ('c'): [1. 0. 0. 0.]
  position 1 ('a'): [0. 1. 0. 0.]
  position 2 ('t'): [0. 0. 1. 0.]
x.shape = (3, 4)    (3 tokens, each a vector in R^4)

STEP 3 — Describe distribution at one position as delta mixture
Suppose position 1 has categorical probs p = [0.1 0.7 0.  0.2]
  (sum = 1.0)

p_data(x) = sum_i p_i * delta(x - e_i)
         = 0.1*delta(x - e_0) + 0.7*delta(x - e_1) + 0.2*delta(x - e_3)

Sampling from this distribution returns one of these vectors:
  e_0 = [1. 0. 0. 0.]   with probability 0.1
  e_1 = [0. 1. 